In [16]:
import numpy as np 
import matplotlib.pyplot as plt
from scipy.constants import c, e, epsilon_0, m_e
from scipy.optimize import minimize

In [17]:
alpha = -1.807364 # 
gamma = 4.169333e1 # m-1

print(f"alpha/gamma = {alpha/gamma:.2f} m")

z_ob = 1994.5 + alpha/gamma #m, where the beam is at waist
z_im = 2005 #m
E0 = 10.0 # GeV
QS = 0.0 #= E - E0 GeV
m12_req = 0.0
m34_req = 0.0
m21_req = 0.0
m_43_req = 0.0

# linac z locations of QS1, QS2 and QS3
z_QS1 = 1996.98249 # [m], middle of quad
z_QS2 = 1999.206615 # [m], middle of quad
z_QS3 = 2001.431049 # [m], middle of quad
LEFF_QS1 = 1 # [m]
LEFF_QS2 = 1 # [m]
LEFF_QS3 = 1 # [m]

OO = np.zeros((2,2))

d1 = (z_QS1 - LEFF_QS1/2) - z_ob 
d2 = (z_QS2 - LEFF_QS2/2) - (z_QS1 + LEFF_QS1/2)
d3 = (z_QS3 - LEFF_QS3/2) - (z_QS2 + LEFF_QS2/2)
d4 = z_im - (z_QS3 + LEFF_QS3/2)


M_01 = np.array([[1, d1], [0, 1]])
M4_01 = np.block([[M_01, OO], [OO, M_01]])
M_02 = np.array([[1, d2], [0, 1]])
M4_02 = np.block([[M_02, OO], [OO, M_02]])
M_03 = np.array([[1, d3], [0, 1]])
M4_03 = np.block([[M_03, OO], [OO, M_03]])
M_04 = np.array([[1, d4], [0, 1]])
M4_04 = np.block([[M_04, OO], [OO, M_04]])

alpha/gamma = -0.04 m


In [18]:
# initial guesses (2012 values)
KQS1_0 = 0.3
KQS2_0 = -0.23
KQS3_0 = 0.3

K_ini = np.array([KQS1_0, KQS2_0, KQS3_0])


def transportError(K):
    # QS1 transport matrix
    k = np.abs(K[0])
    phi = LEFF_QS1*np.sqrt(k)
    M_F = np.array([[np.cos(phi), (1/np.sqrt(k))*np.sin(phi)], [-np.sqrt(k)*np.sin(phi), np.cos(phi)]])
    M_D = np.array([[np.cosh(phi), (1/np.sqrt(k))*np.sinh(phi)], [np.sqrt(k)*np.sinh(phi), np.cosh(phi)]])
    M4_F_1 = np.block([[M_F, OO], [OO, M_D]])

    # QS2 transport matrix
    k = np.abs(K[1])
    phi = LEFF_QS2*np.sqrt(k)
    M_F = np.array([[np.cos(phi), (1/np.sqrt(k))*np.sin(phi)], [-np.sqrt(k)*np.sin(phi), np.cos(phi)]])
    M_D = np.array([[np.cosh(phi), (1/np.sqrt(k))*np.sinh(phi)], [np.sqrt(k)*np.sinh(phi), np.cosh(phi)]])
    M4_D_1 = np.block([[M_D, OO], [OO, M_F]])

    #QS3 transport matrix
    k = np.abs(K[2])
    phi = LEFF_QS3*np.sqrt(k)
    M_F = np.array([[np.cos(phi), (1/np.sqrt(k))*np.sin(phi)], [-np.sqrt(k)*np.sin(phi), np.cos(phi)]])
    M_D = np.array([[np.cosh(phi), (1/np.sqrt(k))*np.sinh(phi)], [np.sqrt(k)*np.sinh(phi), np.cosh(phi)]])
    M4_F_2 = np.block([[M_F, OO], [OO, M_D]])

    # dump line optics
    M4 = M4_04 @ M4_F_2 @ M4_03 @ M4_D_1 @ M4_02 @ M4_F_1 @ M4_01

    chi2 = (M4[0,1]-m12_req)**2 + (M4[2,3]-m34_req)**2 + (M4[0,0] - M4[2,2])**2 #+ 100*(M4[1,0] - m21_req)**2 + 100*(M4[3,2] - m_43_req)**2
    #chi2 = (M4[0,1]-m12_req)**2 + (M4[2,3]-m34_req)**2 + (M4[1,0] - m21_req)**2 + (M4[3,2] - m_43_req)**2

    #chi2 = (M4[0,0] - M4[2,2])**2 + (M4[1,0] - m21_req)**2 + (M4[3,2] - m_43_req)**2

    #chi2 = (M4[0,1] - m12_req)**2 + (M4[1,0] - m21_req)**2 + (M4[0,0] - M4[2,2])**2
    return chi2

result = minimize(transportError, K_ini)

print("Optimized K values:", result.x)

print("Value of chi2 at minimum:", result.fun)
BDES1 =  result.x[0]*(E0 + QS)*LEFF_QS1/0.0299792 # kG
BDES2 =  result.x[1]*(E0 + QS)*LEFF_QS2/0.0299792 # kG
BDES3 =  result.x[2]*(E0 + QS)*LEFF_QS3/0.0299792 # kG

print("Optimized BDES values (kG):", BDES1, BDES2, BDES3)

Optimized K values: [ 0.64824926 -0.69588045  0.5538196 ]
Value of chi2 at minimum: 9.312832204162088e-13
Optimized BDES values (kG): 216.23300694903764 -232.12108816804593 184.73461752759692
